# NLPGym – Guia Completo
Notebook didático com exemplos robustos para Question Answering, Sequence Tagging e Multi-Label Classification usando RL.

## 1. Introdução ao NLPGym
NLPGym converte tarefas de Processamento de Linguagem Natural (NLP) em ambientes de **Reinforcement Learning**.

### Tarefas
| Tarefa | Descrição |
|---|---|
| QA | Pergunta + contexto + opções / agente decide e responde. |
| Sequence Tagging | Token a token (NER, POS). |
| Multi-Label | Texto com vários rótulos possíveis. |

## 2. Passo a Passo
Instalar, criar data pool, ambiente, wrappers, agente, treinar, avaliar.

### 2.1 Wrapper Compatível

In [ ]:
import gymnasium as gym
import gym as ogym
from gymnasium import spaces as gs

class CompatEnv(gym.Env):
    """Wrapper que adapta envs Gym antigos para Gymnasium."""
    def __init__(self, env):
        super().__init__()
        self.env = env
        self.action_space = self._conv(env.action_space)
        self.observation_space = self._conv(env.observation_space)
    def _conv(self, sp):
        if isinstance(sp, ogym.spaces.Box): return gs.Box(low=sp.low, high=sp.high, dtype=sp.dtype)
        if isinstance(sp, ogym.spaces.Discrete): return gs.Discrete(sp.n)
        if isinstance(sp, ogym.spaces.Tuple): return gs.Tuple(tuple(self._conv(s) for s in sp.spaces))
        if isinstance(sp, ogym.spaces.Dict): return gs.Dict({k:self._conv(v) for k,v in sp.spaces.items()})
        return sp
    def reset(self, *, seed=None, **kw):
        obs = self.env.reset()
        return obs, {}
    def step(self, a):
        o,r,d,i = self.env.step(a)
        return o,r,d,False,i

## 3. Exemplos

### 3.1 QA com DQN (CPU)

In [ ]:
from nlp_gym.data_pools.custom_question_answering_pools import QASC
from nlp_gym.envs.question_answering.env import QAEnv
from nlp_gym.envs.question_answering.featurizer import InformedFeaturizer
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
import os

train_pool = QASC.prepare('train')
env_raw = QAEnv(observation_featurizer=InformedFeaturizer())
for s,w in train_pool: env_raw.add_sample(s,w)

vec = DummyVecEnv([lambda: Monitor(CompatEnv(env_raw))])
model = DQN('MlpPolicy', vec, learning_rate=1e-4, batch_size=32, verbose=0)
model.learn(total_timesteps=10000)
os.makedirs('models', exist_ok=True)
model.save('models/dqn_qa_demo')
print('Treino QA concluído')

### 3.2 Sequence Tagging com PPO

In [ ]:
from nlp_gym.data_pools.seq_tagging_pools import CoNLL2003
from nlp_gym.envs.seq_tagging.env import SequenceTaggingEnv
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

train_pool_st = CoNLL2003.prepare('train')
env_st = SequenceTaggingEnv()
for s,_ in train_pool_st: env_st.add_sample(s)
vec_st = DummyVecEnv([lambda: Monitor(CompatEnv(env_st))])
model_st = PPO('MlpPolicy', vec_st, verbose=0)
model_st.learn(total_timesteps=5000)
model_st.save('models/ppo_seq_demo')
print('Sequence Tagging pronto')

### 3.3 Multi-Label com A2C

In [ ]:
from nlp_gym.data_pools.multi_label_pools import Reuters
from nlp_gym.envs.multi_label.env import MultiLabelEnv
from stable_baselines3 import A2C
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

train_pool_ml = Reuters.prepare('train')
env_ml = MultiLabelEnv()
for s,_ in train_pool_ml: env_ml.add_sample(s)
vec_ml = DummyVecEnv([lambda: Monitor(CompatEnv(env_ml))])
model_ml = A2C('MlpPolicy', vec_ml, verbose=0)
model_ml.learn(total_timesteps=5000)
model_ml.save('models/a2c_ml_demo')
print('Multi-Label concluído')

## 4. Treino opcional em GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_model = DQN('MlpPolicy', vec, device='cuda', verbose=0)
    gpu_model.learn(total_timesteps=20000)
    gpu_model.save('models/dqn_qa_gpu_demo')
    print('GPU treino OK')
else:
    print('GPU não disponível')

## 5. Próximos Passos
- Aumentar timesteps
- Reward shaping
- Embeddings ricos
- TensorBoard para monitorar